# Catalog → catalog table copy (overwrite)

Reads **`catalog_copy_config.json`** (list of `schema.table`) plus **source** and **target** catalog names. For each row, runs `spark.table(source_fqtn).write.mode("overwrite").saveAsTable(target_fqtn)` so the destination is replaced.

**Prerequisites:** Unity Catalog (or three-level naming) enabled; cluster identity can **read** the source catalog and **create/overwrite** in the target catalog.

**Config file:** Upload `nxp/load_data/catalog_copy_config.json` to DBFS or a repo path and set **CONFIG_PATH** (absolute path on the driver, e.g. `/dbfs/FileStore/.../catalog_copy_config.json`). Optional widgets override catalog names from JSON.

In [0]:
dbutils.widgets.text(
    "CONFIG_PATH",
    "./catalog_copy_config.json",
    "Absolute path to catalog_copy_config.json on the driver (/dbfs/... or local path)",
)

In [0]:
import json
from pathlib import Path

cfg_path = Path(dbutils.widgets.get("CONFIG_PATH").strip())
if not cfg_path.is_file():
    raise FileNotFoundError(f"Config not found: {cfg_path}")

cfg = json.loads(cfg_path.read_text(encoding="utf-8"))

src_cat = (dbutils.widgets.get("SOURCE_CATALOG") or "").strip() or cfg["source_catalog"].strip()
tgt_cat = (dbutils.widgets.get("TARGET_CATALOG") or "").strip() or cfg["target_catalog"].strip()
mode = cfg.get("load_mode", "overwrite").strip().lower()
fmt = cfg.get("format", "delta").strip().lower()
tables = cfg["tables"]

if not src_cat or not tgt_cat:
    raise ValueError("source_catalog and target_catalog must be set in JSON or widgets")
if mode != "overwrite":
    raise ValueError(f"Only load_mode=overwrite is supported in this notebook; got {mode!r}")

print(f"source_catalog={src_cat}")
print(f"target_catalog={tgt_cat}")
print(f"tables={len(tables)}")

In [0]:
def split_schema_table(entry: str) -> tuple[str, str]:
    s = entry.strip()
    if s.count(".") != 1:
        raise ValueError(f"Each tables[] entry must be schema.table (one dot): {s!r}")
    schema, name = s.split(".", 1)
    if not schema or not name:
        raise ValueError(f"Invalid schema.table: {s!r}")
    return schema, name


def fqtn(catalog: str, schema: str, table: str) -> str:
    return f"{catalog}.{schema}.{table}"


dry = dbutils.widgets.get("DRY_RUN").strip().lower() == "true"

for raw in tables:
    sch, tbl = split_schema_table(raw)
    src = fqtn(src_cat, sch, tbl)
    dst = fqtn(tgt_cat, sch, tbl)
    if dry:
        print(f"DRY_RUN: {src} -> {dst}")
        continue
    df = spark.table(src)
    writer = df.write.mode(mode).format(fmt)
    writer.saveAsTable(dst)
    print(f"OK: {src} -> {dst}")